In [7]:
import pandas as pd
df= pd.read_csv("../DATA/spam_processed.csv")
import pickle
import re
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
with open("../MODELS/tfidf.pkl", "rb") as f:
    tfidf = pickle.load(f)
with open("../MODELS/spam_classifier_model.pkl", "rb") as f:
    model = pickle.load(f)
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(df["c_e"], df["label"], test_size=0.2, random_state=42,stratify=df["label"])
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
with open("../MODELS/logistic_regression_model.pkl", "rb") as f:
    lr_model= pickle.load(f)
from collections import Counter
from sklearn.metrics import classification_report
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()
def trans_text(text):
    text=text.lower()
    text=re.sub(r'[^a-zA-Z0-9 ]', '', text)
    text=text.split()
    text1=[]
    for word in text:
        if word not in stop_words:
            word=ps.stem(word)
            text1.append(word)
    return " ".join(text1)
x_train_tf=tfidf.fit_transform(x_train)
x_test_tf=tfidf.transform(x_test)

[nltk_data] Downloading package stopwords to C:\Users\Mr
[nltk_data]     X\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression()
lr_model.fit(x_train_tf,y_train)
lr_y_pred=lr_model.predict(x_test_tf)
with open("../MODELS/logistic_regression_model.pkl", "wb") as f:
    pickle.dump(lr_model,f)

In [7]:
new_email =" coome let's go to the park and have some fun! "
ch1=new_email
clean_email= trans_text(new_email)
print(clean_email)
new_email_tf=tfidf.transform([clean_email])
print(new_email_tf)
new_email_pred = lr_model.predict(new_email_tf)
print("The new email is classified as:", "Spam" if new_email_pred == 1 else "Ham")

coom let go park fun
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (1, 18929)>
  Coords	Values
  (0, 6196)	0.5715436646136629
  (0, 6589)	0.3243292412954942
  (0, 9166)	0.41270942767406754
  (0, 12228)	0.6307291898981825
The new email is classified as: Ham


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC()
}

for name, model in models.items():

    scores = cross_val_score(
        model,
        x_train_tf,
        y_train,
        cv=5,
        scoring="f1"
    )

    print(name)
    print("Scores:", scores)
    print("Average:", scores.mean())
    print()

Logistic Regression
Scores: [0.78947368 0.76363636 0.75229358 0.77477477 0.76363636]
Average: 0.768762952847936

Naive Bayes
Scores: [0.2962963  0.27848101 0.23376623 0.27848101 0.25641026]
Average: 0.2686869623578484

Linear SVM
Scores: [0.93129771 0.96183206 0.95454545 0.96183206 0.9375    ]
Average: 0.9494014573213047



In [ ]:

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC()
}

for name, model in models.items():

    scores = cross_val_score(
        model,
        x_train_tf,
        y_train,
        cv=5,
        scoring="recall"
    )

    print(name)
    print("Scores:", scores)
    print("Average:", scores.mean())
    print()

Logistic Regression
Scores: [0.65217391 0.61764706 0.60294118 0.63235294 0.61764706]
Average: 0.6245524296675191

Naive Bayes
Scores: [0.17391304 0.16176471 0.13235294 0.16176471 0.14705882]
Average: 0.1553708439897698

Linear SVM
Scores: [0.88405797 0.92647059 0.92647059 0.92647059 0.88235294]
Average: 0.9091645353793691



In [11]:
print(classification_report(y_test, lr_y_pred))
print(confusion_matrix(y_test, lr_y_pred))

              precision    recall  f1-score   support

           0       0.95      1.00      0.98       490
           1       1.00      0.72      0.84        85

    accuracy                           0.96       575
   macro avg       0.98      0.86      0.91       575
weighted avg       0.96      0.96      0.96       575

[[490   0]
 [ 24  61]]
